In [ ]:
from __future__ import annotations
import numpy as np
import random
import copy
import importlib
import matplotlib.pyplot as plt

from typing import Tuple, List
from numpy import array, zeros

# from Big_Class import Big_Class  # already imported one NETfuncs is imported
from User_Variables import User_Variables  # already imported one NETfuncs is imported
from Network_Structure import Network_Structure  # already imported one NETfuncs is imported
from Big_Class import Big_Class
from Network_State import Network_State
from Networkx_Net import Networkx_Net
from Color_Scheme import Color_Scheme
import matrix_functions, functions, statistics, plot_functions, solve, figure_plots, colors

## colors

In [ ]:
importlib.reload(colors)

colors_lst, red, cmap = colors.color_scheme()
cmap

Colorscheme = Color_Scheme(show=True)

# Set up Network

In [ ]:
## Parameters

## task type
# task_type='Iris_classification'
task_type='Regression'

## specify # of nodes
extraNin: int = 0
Ninter: int = 0
extraNout: int = 0

# length of training dataset
iterations = 1200  # number of sampled of p

# measure accuracy every # steps
measure_accuracy_every = 15

supress_prints: bool = True  # whether to print information during training or not
bc_noise: float = 0.0  # noise to dual problem
use_p_tag: bool = True  # use 1 or 2 sampled pressures at every time step
# use_p_tag: bool = False
# include_Power: bool = True
include_Power: bool = False
lam = -80**(1)
lam2 = -5*10**(-3)

stay_sample: int = 2  # how many loop iterations to stay under the same sampled p

random_state = 58

## Networkx sizes
scale: float = 50.0
squish: float = 0.01
    
## User Variables - Keep those since not in use Sep2024

access_interNodes: bool = False  # access and change pressure at interNodes (nodes between input and output) or not
noise_to_extra: bool = False  # add noise to extra outputs 

In [ ]:
## network type
net_type='square'
# net_type='FC'

# for square network
net_height=3
net_length=2
rand_seed=19  # seed for random nodes as inputs, outputs, ground

## task matrix X
M_values: np.ndarray = array([0.15, 0.2, 0.25, 0.1])
    
## # nodes
Nin: int = 1
Nout: int = 1
    
# learning rate
alpha: float = 0.4  # for network combine attempt

# resistance-pressure proportionality factor
gamma: np.ndarray = np.array([1.0])

## method to update resistances - physical property of the system
R_update: str = 'R_propto_dp'
    
add_ground=False

In [ ]:
# resistances initial
# R_vec_i = np.ones(Nin*Nout + Nin + Nout)
R_vec_i = np.array([1, 3, 2, 4, 9, 1, 10])
# R_vec_i = np.array([1, 1, 1])

In [ ]:
import networkx as nx

SQRENET = nx.grid_2d_graph(4, 5, periodic=False, create_using=None)

In [ ]:
SQRENET.nodes

In [ ]:
## Variables class - mostly user choices
Variabs = User_Variables(iterations,\
                         Nin, \
                         extraNin, \
                         Ninter, \
                         Nout, \
                         extraNout, \
                         gamma, \
                         R_update, \
                         use_p_tag, \
                         include_Power, lam, \
                         supress_prints, \
                         bc_noise, \
                         access_interNodes, \
                         task_type, \
                         measure_accuracy_every)
Variabs.assign_alpha_vec(alpha)
print('alpha_vec', Variabs.alpha_vec)
Variabs.create_dataset_and_targets(random_state=random_state, M_values=M_values)
Variabs.create_noise_for_extras()

## Assign input and output nodes a.f.o lattice size and row choice
if net_type == "square":
    inInterOutGround_tuple = matrix_functions.build_input_output_and_ground(Variabs.Nin, Variabs.extraNin, Variabs.Ninter, 
                                                                            Variabs.Nout, Variabs.extraNout, 
                                                                            add_ground=add_ground, net_type=net_type, 
                                                                            seed=rand_seed, net_height=net_height, 
                                                                            net_len=net_length)
else:
    inInterOutGround_tuple = matrix_functions.build_input_output_and_ground(Variabs.Nin, Variabs.extraNin, Variabs.Ninter, 
                                                                            Variabs.Nout, Variabs.extraNout)

In [ ]:
import random as rand
type(rand.sample(range(0, 1 * 2), 2))

In [ ]:
print('input_nodes_arr ', inInterOutGround_tuple[0])
print('output_nodes_arr ', inInterOutGround_tuple[3])
print('ground_nodes_arr ', inInterOutGround_tuple[5])

In [ ]:
## Big Class containing all classes in Network Simulation
BigClass = Big_Class(Variabs)
BigClass.add_Colors(Colorscheme)
## Structure class - build incidence matrices and 1d arrays of edges

Strctr = Network_Structure(inInterOutGround_tuple, net_type, net_height, net_length)
if Ninter >= 1:
    Strctr.build_incidence('partialInter')
else:
    Strctr.build_incidence(net_type)
# Strctr.build_edges()
BigClass.add_Strctr(Strctr)  # add to big class

## Initiate internal flow network state class

State = Network_State(Variabs)
if task_type == 'Iris_classification':
    State.initiate_resistances(BigClass, R_vec_i)
    State.initiate_accuracy_vec(BigClass, measure_accuracy_every)
else:
    State.initiate_resistances(BigClass, R_vec_i)
print('initial resistances', State.R_in_t[0])
BigClass.add_State(State)  # add to big class

## build network graphics class and plot structure

NET = Networkx_Net(scale, squish)
NET.buildNetwork(BigClass)
NET.build_pos_lattice(BigClass, plot=True, node_labels=True)
BigClass.add_NET(NET)  # add to big class

In [ ]:
Strctr.EIEJ_plots

# Iterations

In [ ]:
State.draw_p_in_and_desired(Variabs, 0)  # not used, just for the loop to work.
State.solve_flow_given_problem(BigClass, "measure")
for i in range(Variabs.iterations):
    # These have no meaning. Just need them for loop to work.
    State.draw_p_in_and_desired(Variabs, i+1)
    State.solve_flow_given_problem(BigClass, "measure")
    State.loss_in_t.append([np.zeros(Variabs.Nout), np.zeros(Variabs.Nout)])
    State.loss_norm_in_t.append([np.zeros(Variabs.Nout), np.zeros(Variabs.Nout)])
    State.update_input_dual(BigClass)
    State.update_output_dual(BigClass)
    
    # These are important for iterations
    print('i', i)
    
    State.t += 1
    print('time=', State.t)

    print('solving dual problem')

    State.solve_flow_given_problem(BigClass, "dual", access_inters=access_interNodes)  # measure and don't change resistances
    print('Power', np.sum(State.u**2*State.R_in_t[-1]))
    print('full p', State.p)
    print('updating Rs')
    State.update_Rs(BigClass)
#     print('maximal R', np.max(State.R_in_t[-1]))
#     print('minimal R', np.min(State.R_in_t[-1]))
    print('full resistances', State.R_in_t[-1])
    
    NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
    NET.save_p_reordered(State.p)
    NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
    plot_functions.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
                                R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

In [ ]:
np.sum(State.u**2*State.R_in_t[-1])

In [ ]:
plot_functions.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
                                R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)